In [ ]:
from aligner.networksHandling import helper as helper_networks
from aligner.poissonAlignment import helper as helper_poisson
from aligner.montecarlo import helper as helper_montecarlo
from aligner.analysis import helper as helper_analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
import pickle

In [ ]:
#create directories to store results (if they don't exist)
!mkdir -p results/

## Group-constrained 

Here we have the code for the C. elegans connectome, but the dataset can be changed easily to use the email network (the other one for which we consider groups). We provide the alternative code with a comment right after uploading the data of the networks and the groups. The rest is exactly the same.

In [ ]:
A = helper_networks.load_networks(data_folder="data/processed/celegans/", net_ids=[1,2,3,4,5,6,7,8])
T,N,_ = A.shape

neuron_metadata = pd.read_csv("data/processed/celegans/neuron_metadata.csv", index_col=0) 
groups = np.array(neuron_metadata["location"])

In [ ]:
# A = helper_networks.load_networks(data_folder="data/processed/emails/", net_ids=["07","08","09","10"])
# T,N,_ = A.shape

# user_metadata = pd.read_csv("data/processed/emails/user_metadata.csv", index_col=0) 
# groups = np.array(user_metadata["faculty"])

In [ ]:
A = (A > 0).astype(int)

In [ ]:
#Parameters of the Monte Carlo:
MCsteps = 10000
start_sampling = 50000
freq_sampling = 2

R = 5 #number of runs

## Parallel tempering:
betas = np.array([1.005**n for n in range(-12,1)])
B = len(betas)

In [ ]:
#Initial permutation of the snapshots:
P = helper_montecarlo.initial_permutation_groups(A, groups) #nodes are ordered by strength

In [ ]:
# Ground-truth energy
gtP = [np.arange(N) for t in range(T)] #ground truth permutation

Egt = helper_poisson.calculateEnergy_total(
    A,
    gtP,
    helper_poisson.get_Salpha(A),
    N,
    T
)

In [ ]:
all_results = []

for run in range(R):

    start = time.process_time()

    Esweeps_r, samples, groundState, MCacceptance_rate = (
        helper_montecarlo.sampling_poisson_groups(
            A, P, betas, MCsteps, start_sampling, freq_sampling, groups, ""
        )
    )

    end = time.process_time()

    print(f"Process time (run {run}, s): {end - start}")

    results = {
        "E": Esweeps_r,
        "E_GT": Egt,
        "GS": groundState,
        "Samples": samples,
        "MCacceptance_rate": MCacceptance_rate
    }

    with open(f"celegans_groups_run{run}.pkl", "wb") as file:
        pickle.dump(results, file)

    all_results.append(results)

    # Plot energy sweeps
    plt.figure(figsize=(4, 2))

    for b in range(B):
        plt.plot(Esweeps_r[b], label=round(betas[b], 3))

    plt.plot(
        [Egt for _ in range(len(Esweeps_r[0]))],
        color="red",
        linestyle="dashed",
        label="GT Energy"
    )

    plt.title(f"run={run}")

In [ ]:
# ANALYSIS

file_results = f"celegans_groups_" 
    
for run in range(R):
    plt.figure(figsize=(3,1))
    with open(f"{file_results}run{run}.pkl", "rb") as file:
        results = pickle.load(file)
        all_results.append(results)

    for b in range(B):
        plt.plot(results["E"][b], label=round(betas[b], 3))
    plt.plot([results["E_GT"] for _ in range(len(results["E"][0]))], color="red", linestyle="dashed", label = "Ground truth")

    plt.show()

#sampling average:
S = len(results["Samples"])
samples = np.zeros((S*R, T, N)) 
for run in range(R):
    run_samples = all_results[run]["Samples"]
    samples[run*S:(run+1)*S, :, :] = run_samples  

ensemble = samples
ensemble_node_mapping = helper_analysis.infer_node_mapping(ensemble)
ensemble_node_mapping = helper_analysis.mark_empty_nodes(ensemble_node_mapping, A)
ensemble_node_mapping = helper_analysis.order_node_mapping(ensemble_node_mapping, A, groups) #to sort nodes by desc strngth in the densest snapshot (in this case, A.2)

accuracy = helper_analysis.compute_accuracy(ensemble_node_mapping) #disregarding non-alignable nodes
helper_analysis.show_alignment(ensemble_node_mapping, N, T, True, True, np.arange(T), 
                               f"Inferred node mapping (accuracy={100*accuracy:.2f})", groups)
plt.show()

## No group constrained

Here we have the code for the world trade network, but the dataset can be changed easily to use the mobility network (the other one for which we do not consider groups). We provide the alternative code with a comment right after uploading the data of the networks and the groups. The rest is exactly the same.

In [ ]:
data_ids = [str(year) for year in range(2002, 2023, 2)]
A = helper_networks.load_networks(data_folder="data/processed/worldtrade/", net_ids=data_ids)
T,N,_ = A.shape

In [ ]:
# data_ids = ["2019_01_07","2019_02_04","2019_03_04","2019_04_01","2019_05_06","2019_06_03",
# "2019_07_01","2019_08_05","2019_09_02","2019_10_07","2019_11_04","2019_12_02"]
# A = helper_networks.load_networks(data_folder="data/processed/mobility/", net_ids=data_ids)
# T,N,_ = A.shape

In [ ]:
A = (A > 0).astype(int)

In [ ]:
#Parameters of the Monte Carlo:
MCsteps = 10000
start_sampling = 5000
freq_sampling = 2

R = 5 #number of runs

## Parallel tempering:
betas = np.array([1.005**n for n in range(-12,1)])
B = len(betas)

In [ ]:
#Initial permutation of the snapshots:
P = helper_montecarlo.initial_permutation(A) #nodes are ordered by strength

In [ ]:
# Ground-truth energy
gtP = [np.arange(N) for t in range(T)] #ground truth permutation

Egt = helper_poisson.calculateEnergy_total(
    A,
    gtP,
    helper_poisson.get_Salpha(A),
    N,
    T
)

In [ ]:
all_results = []

for run in range(R):

    start = time.process_time()

    Esweeps_r, samples, groundState, MCacceptance_rate = (
        helper_montecarlo.sampling_poisson(
            A, P, betas, MCsteps, start_sampling, freq_sampling, ""
        )
    )

    end = time.process_time()

    print(f"Process time (run {run}, s): {end - start}")

    results = {
        "E": Esweeps_r,
        "E_GT": Egt,
        "GS": groundState,
        "Samples": samples,
        "MCacceptance_rate": MCacceptance_rate
    }

    with open(f"wtn_run{run}.pkl", "wb") as file:
        pickle.dump(results, file)

    all_results.append(results)

    # Plot energy sweeps
    plt.figure(figsize=(4, 2))

    for b in range(B):
        plt.plot(Esweeps_r[b], label=round(betas[b], 3))

    plt.plot(
        [Egt for _ in range(len(Esweeps_r[0]))],
        color="red",
        linestyle="dashed",
        label="GT Energy"
    )

    plt.title(f"run={run}")

In [ ]:
# ANALYSIS

file_results = f"wtn_" 
    
for run in range(R):
    plt.figure(figsize=(3,1))
    with open(f"{file_results}run{run}.pkl", "rb") as file:
        results = pickle.load(file)
        all_results.append(results)

    for b in range(B):
        plt.plot(results["E"][b], label=round(betas[b], 3))
    plt.plot([results["E_GT"] for _ in range(len(results["E"][0]))], color="red", linestyle="dashed", label = "Ground truth")

    plt.show()

#sampling average:
S = len(results["Samples"])
samples = np.zeros((S*R, T, N)) 
for run in range(R):
    run_samples = all_results[run]["Samples"]
    samples[run*S:(run+1)*S, :, :] = run_samples  

ensemble = samples
ensemble_node_mapping = helper_analysis.infer_node_mapping(ensemble)
ensemble_node_mapping = helper_analysis.mark_empty_nodes(ensemble_node_mapping, A)
ensemble_node_mapping = helper_analysis.order_node_mapping(ensemble_node_mapping, A) #to sort nodes by desc strngth in the densest snapshot (in this case, A.2)

accuracy = helper_analysis.compute_accuracy(ensemble_node_mapping) #disregarding non-alignable nodes
helper_analysis.show_alignment(ensemble_node_mapping, N, T, True, True, np.arange(T), 
                               f"Inferred node mapping (accuracy={100*accuracy:.2f})", None)
plt.show()